In [2]:
import pandas as pd

# Load the dataset
links = pd.read_csv('../data/full_dataset/links.csv')
movies = pd.read_csv('../data/full_dataset/movies.csv')
ratings = pd.read_csv('../data/full_dataset/ratings.csv')
tags = pd.read_csv('../data/full_dataset/tags.csv')

print("Links Dataset shape:", links.shape)
print("Movies Dataset shape:", movies.shape)
print("Ratings Dataset shape:", ratings.shape)
print("Tags Dataset shape:", tags.shape)

Links Dataset shape: (87585, 3)
Movies Dataset shape: (87585, 3)
Ratings Dataset shape: (32000204, 4)
Tags Dataset shape: (2000072, 4)


In [3]:
links.head()

,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0


In [4]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [5]:
tags.head()

,userId,movieId,tag,timestamp
0,22,26479,Kevin Kline,1583038886
1,22,79592,misogyny,1581476297
2,22,247150,acrophobia,1622483469
3,34,2174,music,1249808064
4,34,2174,weird,1249808102


In [6]:
ratings.head()

,userId,movieId,rating,timestamp
0,1,17,4.0,944249077
1,1,25,1.0,944250228
2,1,29,2.0,943230976
3,1,30,5.0,944249077
4,1,32,5.0,943228858


In [7]:
# Basic info and missing values
def df_overview(df, name):
    print(f"{name} shape: {df.shape}")
    print(df.dtypes)
    print("Missing values:")
    print(df.isna().sum())
    print("Duplicate rows:", df.duplicated().sum())
    print('-' * 40)

df_overview(links, 'links')
df_overview(movies, 'movies')
df_overview(ratings, 'ratings')
df_overview(tags, 'tags')

links shape: (87585, 3)
movieId      int64
imdbId       int64
tmdbId     float64
dtype: object
Missing values:
movieId      0
imdbId       0
tmdbId     124
dtype: int64
Duplicate rows: 0
----------------------------------------
movies shape: (87585, 3)
movieId     int64
title      object
genres     object
dtype: object
Missing values:
movieId    0
title      0
genres     0
dtype: int64
Duplicate rows: 0
----------------------------------------
ratings shape: (32000204, 4)
userId         int64
movieId        int64
rating       float64
timestamp      int64
dtype: object
Missing values:
userId       0
movieId      0
rating       0
timestamp    0
dtype: int64
Duplicate rows: 0
----------------------------------------
tags shape: (2000072, 4)
userId        int64
movieId       int64
tag          object
timestamp     int64
dtype: object
Missing values:
userId        0
movieId       0
tag          17
timestamp     0
dtype: int64
Duplicate rows: 0
----------------------------------------


In [8]:
# Ratings distribution and basic stats
print(ratings['rating'].describe())
print(ratings['rating'].value_counts().sort_index())

count    3.200020e+07
mean     3.540396e+00
std      1.058986e+00
min      5.000000e-01
25%      3.000000e+00
50%      3.500000e+00
75%      4.000000e+00
max      5.000000e+00
Name: rating, dtype: float64
rating
0.5     525132
1.0     946675
1.5     531063
2.0    2028622
2.5    1685386
3.0    6054990
3.5    4290105
4.0    8367654
4.5    2974000
5.0    4596577
Name: count, dtype: int64


In [9]:
# Ratings per user and per movie (long-tail check)
user_counts = ratings['userId'].value_counts()
movie_counts = ratings['movieId'].value_counts()

print('Users:', user_counts.shape[0])
print('Movies:', movie_counts.shape[0])
print('Ratings per user (min/median/mean/max):', user_counts.min(), user_counts.median(), user_counts.mean().round(2), user_counts.max())
print('Ratings per movie (min/median/mean/max):', movie_counts.min(), movie_counts.median(), movie_counts.mean().round(2), movie_counts.max())

Users: 200948
Movies: 84432
Ratings per user (min/median/mean/max): 20 73.0 159.25 33332
Ratings per movie (min/median/mean/max): 1 5.0 379.01 102929


In [10]:
# Sparsity estimate
n_users = ratings['userId'].nunique()
n_movies = ratings['movieId'].nunique()
n_ratings = ratings.shape[0]
sparsity = 1 - (n_ratings / (n_users * n_movies))
print(f'Users: {n_users}, Movies: {n_movies}, Ratings: {n_ratings}')
print(f'Sparsity: {sparsity:.6f}')

Users: 200948, Movies: 84432, Ratings: 32000204
Sparsity: 0.998114


In [11]:
# Tag stats
print('Unique tags:', tags['tag'].nunique())
print('Most common tags:')
print(tags['tag'].value_counts().head(20))

Unique tags: 140979
Most common tags:
tag
sci-fi                10996
atmospheric            9589
action                 8473
comedy                 8139
funny                  7467
surreal                7231
visually appealing     7090
based on a book        6617
twist ending           6521
dark comedy            6053
thought-provoking      6000
dystopia               5646
violence               5389
cinematography         5277
romance                5271
murder                 5180
social commentary      5152
stylized               4917
fantasy                4844
psychology             4827
Name: count, dtype: int64


In [12]:
# Coverage checks between datasets
movies_in_ratings = ratings['movieId'].nunique()
movies_in_tags = tags['movieId'].nunique()
movies_in_movies = movies['movieId'].nunique()

print('Movies in ratings:', movies_in_ratings)
print('Movies in tags:', movies_in_tags)
print('Movies in movies table:', movies_in_movies)

missing_movies = set(ratings['movieId'].unique()) - set(movies['movieId'].unique())
print('Movies in ratings but not in movies:', len(missing_movies))

Movies in ratings: 84432
Movies in tags: 51323
Movies in movies table: 87585
Movies in ratings but not in movies: 0


In [13]:
# Time analysis
ratings_ts = ratings.copy()
ratings_ts['datetime'] = pd.to_datetime(ratings_ts['timestamp'], unit='s')
ratings_ts['year'] = ratings_ts['datetime'].dt.year

print(ratings_ts['datetime'].min(), ratings_ts['datetime'].max())
print(ratings_ts['year'].value_counts().sort_index().tail(20))

1995-01-09 11:46:44 2023-10-13 02:29:07
year
2004    1139068
2005    1752573
2006    1141704
2007    1023813
2008    1117071
2009     890696
2010     860073
2011     729301
2012     695493
2013     563301
2014     518537
2015    1743866
2016    1918739
2017    1827953
2018    1391057
2019    1385467
2020    1688159
2021    1239912
2022     913105
2023     794054
Name: count, dtype: int64


In [14]:
# Genre counts
genres = movies['genres'].str.split('|').explode()
print(genres.value_counts())

genres
Drama                 34175
Comedy                23124
Thriller              11823
Romance               10369
Action                 9668
Documentary            9363
Horror                 8654
(no genres listed)     7080
Crime                  6976
Adventure              5402
Sci-Fi                 4907
Animation              4617
Children               4520
Mystery                4013
Fantasy                3851
War                    2325
Western                1696
Musical                1059
Film-Noir               353
IMAX                    195
Name: count, dtype: int64
